# Mistral 7B
"Mistral 7B" (Jiang et al., 2023)

**Idea:** the Llama block (RMSNorm, SwiGLU, RoPE, grouped-query attention) with attention restricted to a **sliding window** of the last $W$ tokens. Each layer sees only $W$ tokens back, but stacking layers lets information travel further, and the KV cache becomes a fixed-size **rolling buffer** instead of growing with the sequence.

### Mistral 7B vs Llama 2
| | Llama 2 7B (`llama2.ipynb`) | Mistral 7B (this notebook) |
|---|---|---|
| Block | RMSNorm, SwiGLU, RoPE | same |
| Attention heads | 32 query, 32 key/value | 32 query, **8 key/value (GQA)** |
| Attention span per layer | all previous tokens | **last $W = 4096$ tokens** |
| Reach after $L$ layers | everything | up to $L \times W$ = 32 x 4096 $\approx$ 131k tokens |
| KV cache | grows with $T$ | **rolling buffer of $W$ entries**, constant memory |
| Attention cost | $O(T^2)$ | $O(T \cdot W)$ |
| Context | 4k | 8k (v0.1) |

GQA, the KV cache and top-p are built in `llama2.ipynb` and are not repeated; attention here is plain multi-head so the window is the only new thing.

Shape flow: `idx (B, T) -> embed (B, T, D) -> [RMSNorm -> attn(RoPE, window W) -> + -> RMSNorm -> SwiGLU -> +] x L -> RMSNorm -> logits (B, T, V)`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data and shared pieces (same as `slm.ipynb`)
Character-level tiny Shakespeare, same split and batch shape as `slm.ipynb`. RMSNorm, RoPE and SwiGLU are copied unchanged; see `slm.ipynb` / `rope.ipynb` for their formulas.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
block_size, batch_size = 128, 32   # T, B

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


def rope_angles(pos, head_dim, base=10000):
    # pos (T,) absolute positions -> cos, sin each (T, head_dim/2); computed on the fly, so any position works
    freqs = 1.0 / base ** (torch.arange(0, head_dim, 2, device=pos.device).float() / head_dim)
    angles = pos.float()[:, None] * freqs[None]
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)


class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden, bias=False)
        self.w3 = nn.Linear(dim, hidden, bias=False)
        self.w2 = nn.Linear(hidden, dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

## 2. Sliding-window attention
### Formula: Sliding-window mask
$$M_{mn} = \begin{cases} 0 & m - W < n \le m \\ -\infty & \text{otherwise} \end{cases}$$
- $m$: query position, $n$: key position, $W$: window size (16 here, 4096 in Mistral 7B)
- token $m$ attends to itself and the $W - 1$ tokens before it: a **band** of width $W$ under the diagonal, instead of the full lower triangle
- attention cost per layer drops from $O(T^2)$ to $O(T \cdot W)$

### Formula: Receptive field after $k$ layers
$$\text{reach}(k) = k\,(W - 1) + 1 \ \text{tokens}$$
- layer 1: position $m$ reads positions $m - W + 1 .. m$. Layer 2 reads *those positions' layer-1 outputs*, each of which already summarizes $W$ tokens, and so on
- same reasoning as stacked convolutions (`convolution.ipynb`): the receptive field grows linearly with depth
- Mistral 7B: $k = 32$, $W = 4096$ gives a theoretical span of ~131k tokens. Distant information arrives only through several hops, so it is weaker than direct attention

In [ ]:
def window_mask(T, W, device=None):
    # -> (T, T) bool, True = allowed: causal AND within the last W positions
    i = torch.arange(T, device=device)
    d = i[:, None] - i[None, :]            # (T, T) m - n
    return (d >= 0) & (d < W)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
ax[0].imshow(window_mask(24, 24), cmap="Blues"); ax[0].set_title("full causal (W >= T)")
ax[1].imshow(window_mask(24, 6), cmap="Blues"); ax[1].set_title("sliding window, W = 6")
for a in ax: a.set_xlabel("key position n"); a.set_ylabel("query position m")
plt.show()


class WindowAttention(nn.Module):
    # x (B, T, D) -> (B, T, D); with a cache, x is ONE new token and cache holds the last W - 1 keys/values
    def __init__(self, dim, num_heads, window):
        super().__init__()
        self.num_heads, self.head_dim, self.window = num_heads, dim // num_heads, window
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, pos, cache=None):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                          # each (B, heads, T, hd)
        cos, sin = rope_angles(pos, self.head_dim)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        if cache is None:                                             # training / full pass: band mask
            att = (q @ k.transpose(-2, -1)) / self.head_dim ** 0.5    # (B, heads, T, T)
            att = att.masked_fill(~window_mask(T, self.window, x.device), float("-inf")).softmax(dim=-1)
            return self.proj((att @ v).transpose(1, 2).reshape(B, T, D)), None

        k, v = cache.update(k, v, pos)                                # (B, heads, <= W, hd): everything in the window
        att = ((q @ k.transpose(-2, -1)) / self.head_dim ** 0.5).softmax(dim=-1)   # no mask: the buffer IS the window
        return self.proj((att @ v).transpose(1, 2).reshape(B, T, D)), cache

## 3. Rolling buffer cache
With a window, keys and values older than $W$ positions are never needed again. So the cache has a **fixed size $W$**, and position $i$ is written to slot $i \bmod W$, overwriting the entry from $W$ steps ago.

### Formula: Rolling buffer
$$\text{slot}(i) = i \bmod W$$
- memory: $W$ entries per layer whatever the sequence length. For a 32k-token sequence with $W = 4096$, 8x less than a full cache
- the slot order is not the time order, and it does not need to be: softmax attention is a sum over the keys, and each key already carries its position through RoPE

In [ ]:
class RollingBuffer:
    # fixed-size KV store for one layer; position i lives in slot i % W
    def __init__(self, window):
        self.window, self.k, self.v, self.filled = window, None, None, 0

    def update(self, k, v, pos):
        # k, v (B, heads, 1, hd) of the new token at absolute position pos -> all valid (k, v) in the buffer
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = torch.zeros(B, H, self.window, hd, device=k.device)
            self.v = torch.zeros(B, H, self.window, hd, device=k.device)
        slot = int(pos[0]) % self.window
        self.k[:, :, slot], self.v[:, :, slot] = k[:, :, 0], v[:, :, 0]     # overwrite the oldest entry
        self.filled = min(self.filled + 1, self.window)
        return self.k[:, :, :self.filled], self.v[:, :, :self.filled]


class Block(nn.Module):
    def __init__(self, dim, num_heads, window):
        super().__init__()
        self.norm1, self.attn = RMSNorm(dim), WindowAttention(dim, num_heads, window)
        self.norm2, self.ffn = RMSNorm(dim), SwiGLU(dim, 64 * ((8 * dim // 3 + 63) // 64))

    def forward(self, x, pos, cache=None):
        a, cache = self.attn(self.norm1(x), pos, cache)
        x = x + a
        return x + self.ffn(self.norm2(x)), cache


class Mistral(nn.Module):
    # idx (B, T) -> logits (B, T, V)
    def __init__(self, vocab_size, window=16, dim=256, depth=6, num_heads=4):
        super().__init__()
        self.window = window
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([Block(dim, num_heads, window) for _ in range(depth)])
        self.norm = RMSNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, idx, caches=None, start_pos=0):
        x = self.embed(idx)                                           # (B, T, D)
        pos = torch.arange(start_pos, start_pos + x.shape[1], device=x.device)
        for i, block in enumerate(self.blocks):
            x, _ = block(x, pos, None if caches is None else caches[i])
        return self.head(self.norm(x))

    def new_caches(self):
        return [RollingBuffer(self.window) for _ in self.blocks]

model = Mistral(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | window {model.window} | layers {len(model.blocks)}")

## 4. Two checks
**Receptive field.** Take the gradient of the *last* position's output with respect to every input embedding. Positions with a non-zero gradient are the ones that can influence it. With $W = 16$ and $k$ layers the reach should be exactly $k \cdot 15 + 1$ tokens.

**Rolling buffer = band mask.** Feeding tokens one at a time through the rolling buffers must give the same logits as one full pass with the sliding-window mask, including after the buffer has wrapped around several times.

In [ ]:
T = 128
x = get_batch("val")[0][:1, :T]
for depth in [1, 2, 4, 6]:
    emb = model.embed(x).detach().requires_grad_(True)                  # (1, T, D)
    h = emb
    pos = torch.arange(T, device=device)
    for block in model.blocks[:depth]:
        h, _ = block(h, pos)
    h[0, -1].sum().backward()                                           # output at the last position
    reach = (emb.grad[0].abs().sum(-1) > 0).sum().item()                # how many inputs it depends on
    print(f"{depth} layer(s): last token depends on {reach:3d} inputs | formula k(W-1)+1 = {min(T, depth * (model.window - 1) + 1)}")

model.eval()
x = get_batch("val")[0][:1, :80]                                        # 80 tokens: the 16-slot buffer wraps 5 times
with torch.no_grad():
    full = model(x)                                                     # (1, 80, V) band-masked full pass
    caches = model.new_caches()
    steps = torch.cat([model(x[:, t:t + 1], caches, start_pos=t) for t in range(x.shape[1])], dim=1)
print("max |full pass - rolling buffer|:", (full - steps).abs().max().item())
print("buffer shape per layer:", tuple(caches[0].k.shape), "after 80 tokens")
model.train()

## 5. Training
Same loop as `slm.ipynb`. The only difference from that model is that each layer sees 16 characters back instead of 128, so compare the val loss with `slm.ipynb`'s to see what the window costs at this scale.

In [ ]:
max_iters = 3000
eval_every = 500
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

def loss_fn(model, x, y):
    return F.cross_entropy(model(x).reshape(-1, vocab_size), y.reshape(-1))

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    loss = loss_fn(model, *get_batch("train"))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 6. Generation with constant memory
One token per step through the rolling buffers. The sequence can run far past the training length of 128: memory stays at $W = 16$ entries per layer, and RoPE angles are computed for whatever position comes next.

(Mistral's code also **pre-fills** the cache with a known prompt in chunks of size $W$ rather than token by token; here the prompt is simply fed one token at a time.)

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=1000, temperature=0.8):
    model.eval()
    caches = model.new_caches()
    ids = encode(start)
    for t, tok in enumerate(ids):                                        # prompt, token by token
        logits = model(torch.tensor([[tok]], device=device), caches, start_pos=t)
    for t in range(len(ids), len(ids) + n):
        nxt = torch.multinomial(F.softmax(logits[:, -1] / temperature, dim=-1), 1)   # (1, 1)
        ids.append(nxt.item())
        logits = model(nxt, caches, start_pos=t)
    model.train()
    return decode(ids), caches

out, caches = generate(model)
print(out)
print(f"\ngenerated {len(out)} characters | cache per layer still {tuple(caches[0].k.shape)}")

## Notes
- Mistral 7B: $L = 32$, $D = 4096$, 32 heads with 8 key/value heads, SwiGLU hidden 14336, 32k vocabulary, $W = 4096$. It outperformed Llama 2 13B on most benchmarks at about half the size; the paper attributes this to training, not to the window.
- Sliding-window attention was dropped in later releases: Mistral 7B v0.2 and newer use full attention with a 32k context. With flash attention and GQA, full attention over tens of thousands of tokens became affordable, and direct access to distant tokens is worth more than the savings.
- The idea of local attention is older: Longformer and BigBird (sliding window + a few global tokens) used it for long-document encoders.
- The receptive-field argument is an upper bound. A fact $5W$ tokens back has to be relayed through at least 5 layers, each time competing for space in the residual stream; tasks that need exact long-range lookup suffer first.
- The rolling buffer works because of two properties: softmax attention is permutation-invariant over the keys, and RoPE stores position inside each rotated key rather than in its slot.